### Install requiremnts

In [1]:
!pip install -r ../requirements.txt

In [32]:
import sys
sys.path.append('../')
import os

from validate_birdset import load_model, to_n_hot, test
import numpy as np
import requests
import zipfile
import glob
import pandas as pd
from datasets import Dataset
from functools import partial
from utils.transform import ValTransformBeans
from utils.event_decoder import EventDecoder
import torch
import json
import gdown

DEVICE ='cuda'
DOWN_TASKS_ACC = ["walkins", "bats", "cbi", "dogs", "humburgdb", "speech"]
DOWN_TASKS_MAP = ["enabirds", "hiceas", "rfcx", "hainan gibbons"]

# Change this to the directory where the BEANS dataset is stored.
# To download BEANS, see: https://github.com/earthspecies/beans
BEANS_ROOT_PATH = "<PATH TO BEANS>"

#### Download checkpoints

In [33]:
def evaluate_beans_classification(ckpts, down_task, device='cuda'):
        models = []
        configs = []

        # ---------------------------------------------------------
        # Load models and their configs from checkpoints
        # ---------------------------------------------------------
        for ckpt in ckpts:
            m, cfg = load_model(ckpt, device=device)
            models.append(m)
            cfg.train.dataset_name = down_task
            configs.append(cfg)

        config = configs[0]
        results = dict()
    
        # ---------------------------------------------------------
        # Load dataset metadata and create validation dataframe
        # ---------------------------------------------------------

        # Different datasets have different metadata file structures
        if down_task == "HumBugDB":
            train_df = pd.read_csv(f"{BEANS_ROOT_PATH}/data/{down_task}/data/metadata/train.csv")
            val_df = pd.read_csv(f"{BEANS_ROOT_PATH}/data/{down_task}/data/metadata/test.csv")
        
        elif down_task == "speech_commands":
            train_df = pd.read_csv(f"{BEANS_ROOT_PATH}/data/{down_task}/train.csv")
            val_df = pd.read_csv(f"{BEANS_ROOT_PATH}/data/{down_task}/test.csv")
        
        else:
            train_df = pd.read_csv(f"{BEANS_ROOT_PATH}/data/{down_task}/annotations.train.csv")
            val_df = pd.read_csv(f"{BEANS_ROOT_PATH}/data/{down_task}/annotations.test.csv")

        # ---------------------------------------------------------
        # Prepare label mapping
        # ---------------------------------------------------------
        class_names = sorted(list(set(train_df.label.tolist())))

        # Convert label names into numeric label indices
        val_df['labels'] = val_df['label'].apply(lambda x: class_names.index(x))
        val_df["filepath"] = val_df["path"].apply(lambda x: f"{BEANS_ROOT_PATH}/{x}")

        config.train.label_map = {class_names[i]: i for i in range(len(class_names))}
        config.train.num_classes = len(class_names)

        # Convert pandas dataframe to HuggingFace Dataset
        val_data = Dataset.from_pandas(val_df)
    
        # ---------------------------------------------------------
        # Convert labels to N-hot format
        # ---------------------------------------------------------
        val_data = val_data.map(
        partial(to_n_hot,
                num_classes=config.train.num_classes,
                ),
        batched=True,
        batch_size=300,
        load_from_cache_file=False,
        num_proc=1,
        )

        # ---------------------------------------------------------
        # Define validation transformation pipeline
        # ---------------------------------------------------------
        val_transform = ValTransformBeans(
        config=config,
        train=False,
        event_decoder=EventDecoder(extracted_interval=config.event_decoder.val.extracted_interval,
                                   sample_rate=config.frontend.sample_rate)
        )

        val_data.set_transform(val_transform)

        # ---------------------------------------------------------
        # Create PyTorch DataLoader for validation dataset
        # ---------------------------------------------------------
        val_loader = torch.utils.data.DataLoader(
            val_data,
            num_workers=config.train.num_workers,
            batch_size=config.train.batch_size,
            drop_last=False,
            shuffle=False
        )
    
        # Map class names to label IDs using the config label map
        label2id = {k: v for k, v in configs[0].train.label_map.items()}

        # Only test on classes relevant for the current dataset
        relevant = [label2id[c] for c in class_names]

        # Store metrics for each model (for later averaging)
        metrics = {
                   "top1_acc": []
        }

        # ---------------------------------------------------------
        # test each model on the test dataset
        # ---------------------------------------------------------
        for model, _ in zip(models, configs):
            # Run evaluation
            _, _, top1_acc = test(model, val_loader, relevant, center_5s=False, device=device)

            # Store metrics
            metrics["top1_acc"].append(top1_acc)

        # ---------------------------------------------------------
        # Aggregate metrics across all checkpoints/models
        # ---------------------------------------------------------
        results[down_task] = {key: float(np.mean(values)) for key, values in metrics.items()}
        print(results)

In [34]:
def load_json_dataset(json_path, config, labels, unknown_label=None, window_width=2, window_shift=1):
    # Open the JSON dataset file (each line contains one audio sample description)
    with open(json_path) as f:
        xs = []
        ys = []
        # Iterate through each line (one recording per line)
        for line in f:
            data = json.loads(line)

            # Path to the audio file
            wav_path = data['path']

            # Total duration of the recording
            length = data['length']

            # Compute how many sliding windows can be extracted
            num_windows = int((length - window_width) / window_shift) + 1

            # ---------------------------------------------------------
            # Slide a window across the audio recording
            # ---------------------------------------------------------
            for window_id in range(num_windows):
                # Compute start and end time of the window
                st, ed = window_id * window_shift, window_id * window_shift + window_width
                # Store window boundaries
                offset_st, offset_ed = st, ed
                # Save input sample (audio path + time window)
                xs.append((wav_path, offset_st, offset_ed))
                # Initialize label vector (multi-label classification)
                y = torch.zeros(len(labels))

                # ---------------------------------------------------------
                # Check overlap between window and annotated events
                # --------------------------------------------------------
                for anon in data['annotations']:
                    if anon['label'] not in labels:
                        if unknown_label is not None:
                            # Assign unknown labels to a predefined class
                            label_id = labels.index(unknown_label)
                        else:
                           raise KeyError(f"Unknown label: {anon['label']}")
                    else:
                        # Get class index
                        label_id = labels.index(anon['label'])

                    # ---------------------------------------------------------
                    # Check if annotation overlaps with the current window
                    # ---------------------------------------------------------

                    # Case 1: annotation start or end falls inside the window
                    if (st <= anon['st'] <= ed) or (st <= anon['ed'] <= ed):
                        denom = min(ed - st, anon['ed'] - anon['st'])
                        if denom == 0:
                            continue
                        overlap = (min(ed, anon['ed']) - max(st, anon['st'])) / denom
                        if overlap > .2:
                            y[label_id] = 1

                    # Case 2: annotation fully contains the window
                    if anon['st'] <= st and ed <= anon['ed']:
                        y[label_id] = 1
                # Save label vector for this window
                ys.append(y)
    return xs, ys

In [35]:
def evaluate_beans_detection(ckpts, down_task, device='cuda'):
        models = []
        configs = []

        # ---------------------------------------------------------
        # Load models and their configs from checkpoints
        # ---------------------------------------------------------
        for ckpt in ckpts:
            m, cfg = load_model(ckpt, device=device)
            models.append(m)
            cfg.train.dataset_name = down_task
            configs.append(cfg)

        config = configs[0]
        results = dict()

        # ---------------------------------------------------------
        # Define dataset-specific parameters
        # source: https://github.com/earthspecies/beans/blob/main/datasets.yml
        # ---------------------------------------------------------
        if down_task == "enabirds":
            labels = ["AMCR", "AMGO", "AMRE", "AMRO", "BAWW", "BCCH", "BGGN", "BHCO", "BHVI", "BLJA", "BTNW", "BWWA",
                     "CARW", "COYE", "EATO", "HETH", "HOWA", "KEWA", "LOWA", "NOCA", "NOFL", "OVEN", "RBGR", "RBWO",
                     "RCKI", "REVI", "SCTA", "SWTH", "TUTI", "WBNU", "WITU", "WOTH", "YBCU", "OTHR"]
            unknown_label = "OTHR"
            window_size = 2
            window_shift = 1

        elif down_task == "hiceas":
            labels =  [1]
            unknown_label = None
            window_size = 10 
            window_shift = 5

        elif down_task == "rfcx":
            labels = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
            unknown_label = None
            window_size = 10
            window_shift = 5

        elif down_task == "hainan_gibbons":
            labels= [1, 2, 3]
            unknown_label= 1
            window_size = 4
            window_shift= 2


        config.train.num_classes = len(labels)

        # ---------------------------------------------------------
        # Load test dataset from JSON annotations
        # ---------------------------------------------------------
        val_xs, val_ys = load_json_dataset(f"{BEANS_ROOT_PATH}/data/{down_task}/test.jsonl", config, labels, unknown_label, window_size, window_shift)

        # ---------------------------------------------------------
        #  Convert dataset into a pandas-style dictionary
        # ---------------------------------------------------------
        val_df = {"filepath": [], "labels": [], "start_time": [], "end_time": []}
        for i, item in enumerate(val_xs):
            val_df['filepath'].append(f"{BEANS_ROOT_PATH}/{item[0]}")
            val_df['start_time'].append(item[1])
            val_df['end_time'].append(item[2])
            val_df['labels'].append(val_ys[i].numpy())
            
        # Convert dictionary to HuggingFace Dataset
        val_data = Dataset.from_pandas(pd.DataFrame.from_dict(val_df))

        config.train.label_map = {labels[i]: i for i in range(len(labels))}

        # ---------------------------------------------------------
        # Create validation transform pipeline
        # ---------------------------------------------------------
        val_transform = ValTransformBeans(
        config=config,
        train=False,
        event_decoder=EventDecoder(extracted_interval=config.event_decoder.val.extracted_interval,
                                   sample_rate=config.frontend.sample_rate)
        )

        val_data.set_transform(val_transform)

        # ---------------------------------------------------------
        # Create PyTorch DataLoader
        # ---------------------------------------------------------
        val_loader = torch.utils.data.DataLoader(
            val_data,
            num_workers=config.train.num_workers,
            batch_size=config.train.batch_size,
            drop_last=False,
            shuffle=False
        )
    
        # Map class names to label IDs using the config label map
        label2id = {k: v for k, v in configs[0].train.label_map.items()}

        # Only test on classes relevant for the current dataset
        relevant = [label2id[c] for c in labels]

        # Store metrics for each model (for later averaging)
        metrics = {
                   "cmap": [],
                   }

        # ---------------------------------------------------------
        # test each model on the test dataset
        # ---------------------------------------------------------
        for model, _ in zip(models, configs):
            # Run evaluation
            _, cmap, _ = test(model, val_loader, relevant, center_5s=False, device=device)

            # Store metrics
            metrics["cmap"].append(cmap)

        # ---------------------------------------------------------
        # Aggregate metrics across all checkpoints/models
        # ---------------------------------------------------------
        results[down_task] = {key: float(np.mean(values)) for key, values in metrics.items()}
        print(results)

In [36]:
# down task: watkins
if not os.path.exists('../ckpts/beans/walkins'):
    gdown.download_folder('https://drive.google.com/drive/folders/1VFkbNe1vtKm98tMMxendx6tvkeIh9fhz?usp=sharing', output="../ckpts/beans/")

# get ckpts paths 
ckpts = glob.glob('../ckpts/beans/walkins/*')

evaluate_beans_classification(ckpts, "watkins", device=DEVICE)
    

Retrieving folder contents


Retrieving folder 1RmR_uHmZ66G59By5v2CAk7qki9BxQVx5 walkins_eca_nfnet_l1_2026-03-14_224546
Retrieving folder 1DsAD8rQVQ3dZG_48SZJsYd_qu6D41Ia4 models
Processing file 1UQqDmDHlTgO-n2TBwWewNLaXKcmWnkpt model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1UQqDmDHlTgO-n2TBwWewNLaXKcmWnkpt
From (redirected): https://drive.google.com/uc?id=1UQqDmDHlTgO-n2TBwWewNLaXKcmWnkpt&confirm=t&uuid=f797104e-3340-4bb4-aacf-4726ca5c9aad
To: /home/bellafkir/Documents/sa4birds-main/ckpts/beans/walkins/walkins_eca_nfnet_l1_2026-03-14_224546/models/model.pth
100%|██████████| 230M/230M [00:02<00:00, 112MB/s]  
Download completed
2026-05-04 16:32:29,718 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 16:32:37,836 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 16:32:37,849 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 6/6 [00:04<00:00,  1.34it/s]

{'watkins': {'top1_acc': 0.8466076850891113}}


In [37]:
# down task: bats 
if not os.path.exists('../ckpts/beans/bats'):
    gdown.download_folder('https://drive.google.com/drive/folders/1kj5JhGWqpk7HodMmLgAsRZRMvNvWRQF-?usp=sharing', output="../ckpts/beans/")

# get ckpts paths
ckpts = glob.glob('../ckpts/beans/bats/*')

evaluate_beans_classification(ckpts, "egyptian_fruit_bats", device=DEVICE)
    

Retrieving folder contents


Retrieving folder 1owRfFmpEZvWkVNau9VrZbYZrEnwmmPuJ egyptian_fruit_bats_eca_nfnet_l1_2026-03-14_153205
Retrieving folder 1t3WYT5vZAKd1AuqVUHK-9QQVtfsMQ5FY models
Processing file 1ASepN5CYTsvPZW6esUz4PD-rYClAJHH4 model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1ASepN5CYTsvPZW6esUz4PD-rYClAJHH4
From (redirected): https://drive.google.com/uc?id=1ASepN5CYTsvPZW6esUz4PD-rYClAJHH4&confirm=t&uuid=1561db53-f6a2-4156-aa08-9fa5b8cf2dbc
To: /home/bellafkir/Documents/sa4birds-main/ckpts/beans/bats/egyptian_fruit_bats_eca_nfnet_l1_2026-03-14_153205/models/model.pth
100%|██████████| 229M/229M [00:02<00:00, 94.1MB/s] 
Download completed
2026-05-04 16:37:06,314 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 16:37:06,471 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 16:37:06,476 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 32/32 [00:50<00:00,  1.59s/it]

{'egyptian_fruit_bats': {'top1_acc': 0.8025000095367432}}


In [38]:
# down task: cbi 
if not os.path.exists('../ckpts/beans/cbi'):
    gdown.download_folder('https://drive.google.com/drive/folders/1AeAjlhSKWqn8lkMmaIWF0kIJ2gx02IJi?usp=sharing', output="../ckpts/beans/")

# get ckpts paths 
ckpts = glob.glob('../ckpts/beans/cbi/*')

evaluate_beans_classification(ckpts, "cbi", device=DEVICE)

Retrieving folder contents


Retrieving folder 11n-Fl40xi_hZWm8LDbw_3hUd2z8AALg4 cbi_eca_nfnet_l1_2026-03-14_231555
Retrieving folder 1sUYF70kBw3MimAdQq3dB7RoCeL68j65p models
Processing file 10xy4XlpYwW3e0V3d2p0F-QIuOihMrU6C model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=10xy4XlpYwW3e0V3d2p0F-QIuOihMrU6C
From (redirected): https://drive.google.com/uc?id=10xy4XlpYwW3e0V3d2p0F-QIuOihMrU6C&confirm=t&uuid=a94ca9ce-28ae-49c8-b80a-8c59afb7bc2f
To: /home/bellafkir/Documents/sa4birds-main/ckpts/beans/cbi/cbi_eca_nfnet_l1_2026-03-14_231555/models/model.pth
100%|██████████| 236M/236M [00:02<00:00, 100MB/s]  
Download completed
2026-05-04 16:38:49,756 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 16:38:49,876 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 16:38:49,887 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 57/57 [00:38<00:00,  1.48it/s]


{'cbi': {'top1_acc': 0.8149171471595764}}


In [39]:
# down task: dogs 
if not os.path.exists('../ckpts/beans/dogs'):
    gdown.download_folder('https://drive.google.com/drive/folders/1Zn2g9d3y5OHrgAoq384-y-hjvGjH9AER?usp=sharing', output="../ckpts/beans/")


# get ckpts paths 
ckpts = glob.glob('../ckpts/beans/dogs/*')

evaluate_beans_classification(ckpts, "dogs", device=DEVICE)
    

Retrieving folder contents


Retrieving folder 1B40xDUnkaA_3WE3u3u4CIUI7hiCPH6fy dogs_eca_nfnet_l1_2026-03-14_175151
Retrieving folder 17tE6vW1O2GGSdg30Vp7d7ZnLbbbHRtZn models
Processing file 13QQMEzBBxyIFdYpF-ZpsUWoBc02ZtFfR model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=13QQMEzBBxyIFdYpF-ZpsUWoBc02ZtFfR
From (redirected): https://drive.google.com/uc?id=13QQMEzBBxyIFdYpF-ZpsUWoBc02ZtFfR&confirm=t&uuid=65d8b0ba-411a-4503-ab2a-0185f9ac07bd
To: /home/bellafkir/Documents/sa4birds-main/ckpts/beans/dogs/dogs_eca_nfnet_l1_2026-03-14_175151/models/model.pth
100%|██████████| 229M/229M [00:02<00:00, 112MB/s]  
Download completed
2026-05-04 16:39:41,658 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 16:39:41,776 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 16:39:41,780 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 3/3 [00:06<00:00,  2.15s/it]

{'dogs': {'top1_acc': 0.9784172773361206}}


In [40]:
# down task: humbugdb 
if not os.path.exists('../ckpts/beans/humbugdb'):
    gdown.download_folder('https://drive.google.com/drive/folders/1pXl_0LNbdvUqAVvx-lUotlePCcXnpa6Y?usp=sharing', output="../ckpts/beans/")


# get ckpts paths 
ckpts = glob.glob('../ckpts/beans/humbugdb/*')

evaluate_beans_classification(ckpts, "HumBugDB", device=DEVICE)
    

Retrieving folder contents


Retrieving folder 1siidUKqmhTgHIzMp6WqPnaE0mxGWAuM1 HumBugDB_eca_nfnet_l1_2026-03-14_180124
Retrieving folder 1q_v8KkQd2o3x7nsqbZHAo1Udv1d6x0fv models
Processing file 10y33gTpHMmiGMSiSdR1umcJbVuSF1bYH model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=10y33gTpHMmiGMSiSdR1umcJbVuSF1bYH
From (redirected): https://drive.google.com/uc?id=10y33gTpHMmiGMSiSdR1umcJbVuSF1bYH&confirm=t&uuid=9e16282c-1633-4e64-8cdc-f7a9a3d5f60f
To: /home/bellafkir/Documents/sa4birds-main/ckpts/beans/humbugdb/HumBugDB_eca_nfnet_l1_2026-03-14_180124/models/model.pth
100%|██████████| 229M/229M [00:02<00:00, 106MB/s]  
Download completed
2026-05-04 16:40:38,708 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 16:40:38,828 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 16:40:38,840 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 30/30 [00:23<00:00,  1.27it/s]


{'HumBugDB': {'top1_acc': 0.7902097702026367}}


In [41]:
# down task: speech 
if not os.path.exists('../ckpts/beans/speech'):
    gdown.download_folder('https://drive.google.com/drive/folders/1GJUWyr6TGjTkTjm3wehvLW4jZmCu8I5k?usp=sharing', output="../ckpts/beans/")

# get ckpts paths
ckpts = glob.glob('../ckpts/beans/speech/*')

evaluate_beans_classification(ckpts, "speech_commands", device=DEVICE)

Retrieving folder contents


Retrieving folder 1Ej9jy5Hf-ApSUE8ibToUAYthOsL4eldK speech_commands_eca_nfnet_l1_2026-03-14_200735
Retrieving folder 1ncQ-90bCilGwT0NDFXJ3vSb2BGrcoZ9F models
Processing file 1Hbu0s-FnAryKhNkyGZpn6HsuBIAXgaLb model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1Hbu0s-FnAryKhNkyGZpn6HsuBIAXgaLb
From (redirected): https://drive.google.com/uc?id=1Hbu0s-FnAryKhNkyGZpn6HsuBIAXgaLb&confirm=t&uuid=136f65cd-ff22-4566-bfba-415b4b555b7e
To: /home/bellafkir/Documents/sa4birds-main/ckpts/beans/speech/speech_commands_eca_nfnet_l1_2026-03-14_200735/models/model.pth
100%|██████████| 230M/230M [00:02<00:00, 105MB/s]  
Download completed
2026-05-04 16:41:19,679 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 16:41:19,843 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 16:41:19,856 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 172/172 [00:15<00:00, 10.95it/s]


{'speech_commands': {'top1_acc': 0.8526124358177185}}


In [42]:
# down task: enabirds 
if not os.path.exists('../ckpts/beans/enabirds'):
    gdown.download_folder('https://drive.google.com/drive/folders/1CBz7tJz7HWqTB8wwQbOcsts7c3nkIM6P?usp=sharing', output="../ckpts/beans/")

# get ckpts paths 
ckpts = glob.glob('../ckpts/beans/enabirds/*')

evaluate_beans_detection(ckpts, "enabirds", device=DEVICE)

Retrieving folder contents


Retrieving folder 1wZRm2CsY8PFic2sa4mBkmGAiLI7nMgmN enabirds_eca_nfnet_l1_2026-03-14_181011
Retrieving folder 1Fl4cVxuUO0fjKxIqo5w9N7d3pR1hbYko models
Processing file 11r284MDTelhuZ89NK9fq_lxhoFsIybm4 model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=11r284MDTelhuZ89NK9fq_lxhoFsIybm4
From (redirected): https://drive.google.com/uc?id=11r284MDTelhuZ89NK9fq_lxhoFsIybm4&confirm=t&uuid=e5730b65-03d3-4dcd-bd2e-a22f31d4184c
To: /home/bellafkir/Documents/sa4birds-main/ckpts/beans/enabirds/enabirds_eca_nfnet_l1_2026-03-14_181011/models/model.pth
100%|██████████| 230M/230M [00:02<00:00, 93.7MB/s] 
Download completed
2026-05-04 16:43:08,174 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 16:43:08,311 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 16:43:08,325 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 71/71 [00:10<00:00,  6.63it/s]

{'enabirds': {'cmap': 0.7615470559040949}}


In [43]:
# down task: hiceas 
if not os.path.exists('../ckpts/beans/hiceas'):
    gdown.download_folder('https://drive.google.com/drive/folders/14Al2-wScMXgX8JDaqxXIb8Y4rEk7HymL?usp=sharing', output="../ckpts/beans/")

# get ckpts paths
ckpts = glob.glob('../ckpts/beans/hiceas/*')

evaluate_beans_detection(ckpts, "hiceas", device=DEVICE)

Retrieving folder contents


Retrieving folder 1s-AM4n73P5RL8yx1Hkx9tiiiaVeX0MGE hiceas_eca_nfnet_l1_2026-03-14_184118
Retrieving folder 1rJ_MaKGGQXPHdSbPvYIY27w_79CUdXz9 models
Processing file 1WwpA0RJ3rfkKN-fxiGpXympZ3bK0Mjo9 model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1WwpA0RJ3rfkKN-fxiGpXympZ3bK0Mjo9
From (redirected): https://drive.google.com/uc?id=1WwpA0RJ3rfkKN-fxiGpXympZ3bK0Mjo9&confirm=t&uuid=1aa3a9f1-952f-429c-b3b9-5e6cdff3817d
To: /home/bellafkir/Documents/sa4birds-main/ckpts/beans/hiceas/hiceas_eca_nfnet_l1_2026-03-14_184118/models/model.pth
100%|██████████| 229M/229M [00:02<00:00, 108MB/s]  
Download completed
2026-05-04 16:44:09,170 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 16:44:09,298 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 16:44:09,314 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 24/24 [00:17<00:00,  1.40it/s]

{'hiceas': {'cmap': 0.6033437344325436}}


In [44]:
# down task: rfcx 
if not os.path.exists('../ckpts/beans/rfcx'):
    gdown.download_folder('https://drive.google.com/drive/folders/1UolTMpZtT8tFy6mXAZPOWRaTgq-wACVb?usp=sharing', output="../ckpts/beans/")

# get ckpts paths 
ckpts = glob.glob('../ckpts/beans/rfcx/*')

evaluate_beans_detection(ckpts, "rfcx", device=DEVICE)

Retrieving folder contents


Retrieving folder 1Y4U0zjtRAXPW5CS6AJHtFy0VCMw3rMRh rfcx_eca_nfnet_l1_2026-03-14_185350
Retrieving folder 1xE2YSfLOZotD2d6Yer0Xa5WdRcoVaAaj models
Processing file 1XGB52_dSag3z2kd-U7fGd_tzVlq4U2XQ model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1XGB52_dSag3z2kd-U7fGd_tzVlq4U2XQ
From (redirected): https://drive.google.com/uc?id=1XGB52_dSag3z2kd-U7fGd_tzVlq4U2XQ&confirm=t&uuid=feca54e7-1282-42df-b9c1-bc5820a51267
To: /home/bellafkir/Documents/sa4birds-main/ckpts/beans/rfcx/rfcx_eca_nfnet_l1_2026-03-14_185350/models/model.pth
100%|██████████| 230M/230M [00:02<00:00, 109MB/s]  
Download completed
2026-05-04 16:45:07,506 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 16:45:07,628 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 16:45:07,640 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 163/163 [01:51<00:00,  1.46it/s]


{'rfcx': {'cmap': 0.20168134723547437}}


In [45]:
# down task: hainan_gibbons 
if not os.path.exists('../ckpts/beans/gibbons'):
    gdown.download_folder('https://drive.google.com/drive/folders/1LODD8Jh4yqNYO23ki8wZg7hGTTIFAW-n?usp=sharing', output="../ckpts/beans/")

# get ckpts paths
ckpts = glob.glob('../ckpts/beans/gibbons/*')

evaluate_beans_detection(ckpts, "hainan_gibbons", device=DEVICE)

Retrieving folder contents


Retrieving folder 1V2gNiicuvuVVokSEDg-33ViLZoCCkilf hainan_gibbons_eca_nfnet_l1_2026-03-14_193229
Retrieving folder 1RL6leh_Gd1WL3lLwkf9o7Yilq-Fe9cYB models
Processing file 1Gp3yUFqTiA59CDsMIENgj-TlEy5jppzt model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1Gp3yUFqTiA59CDsMIENgj-TlEy5jppzt
From (redirected): https://drive.google.com/uc?id=1Gp3yUFqTiA59CDsMIENgj-TlEy5jppzt&confirm=t&uuid=40ae9870-1c51-4ef0-adbf-3297b5224d12
To: /home/bellafkir/Documents/sa4birds-main/ckpts/beans/gibbons/hainan_gibbons_eca_nfnet_l1_2026-03-14_193229/models/model.pth
100%|██████████| 229M/229M [00:02<00:00, 107MB/s]  
Download completed
2026-05-04 16:47:05,196 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 16:47:05,315 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 16:47:05,332 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
Testing: 100%|██████████| 290/290 [00:27<00:00, 10.64it/s]


{'hainan_gibbons': {'cmap': 0.5162510269030967}}
